# Exercise: Implementing the GPT Training Loop

Welcome! In this exercise, you'll build the core components of a training orchestrator, much like the `Trainer` class in Andrej Karpathy's nanoGPT. This is a crucial step in bringing your model to life, enabling it to learn from data.

You've already learned the theory behind the training loop. Now it's time to put that knowledge into practice!

**In this exercise, you will implement:**
*   A method to efficiently sample random batches of data for training.
*   The logic for a single training step: forward pass, loss calculation, backward pass, and parameter update.
*   The main training loop that orchestrates these components over many iterations.

Let's get started!

In [ ]:
import torch
import torch.nn as nn
from torch.nn import functional as F
from dataclasses import dataclass

# Set a seed for reproducibility
torch.manual_seed(42)

# --- Helper Code (Students should NOT modify this) ---

@dataclass
class TrainerConfig:
    """Configuration for the Trainer."""
    max_iters: int = 100
    batch_size: int = 4
    block_size: int = 8
    learning_rate: float = 1e-3

class DummyGPT(nn.Module):
    """A simple model for demonstration purposes."""
    def __init__(self, vocab_size, n_embd, block_size):
        super().__init__()
        self.block_size = block_size
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        tok_emb = self.token_embedding_table(idx) # (B, T, C)
        pos = torch.arange(0, T, dtype=torch.long, device=idx.device)
        pos_emb = self.position_embedding_table(pos) # (T, C)
        x = tok_emb + pos_emb # (B, T, C)
        logits = self.lm_head(x) # (B, T, vocab_size)

        loss = None
        if targets is not None:
            B, T, C = logits.shape
            logits_flat = logits.view(B*T, C)
            targets_flat = targets.view(B*T)
            loss = F.cross_entropy(logits_flat, targets_flat)

        return logits, loss

    def configure_optimizers(self, config: TrainerConfig) -> torch.optim.Optimizer:
        """Creates an AdamW optimizer for the model's parameters."""
        return torch.optim.AdamW(self.parameters(), lr=config.learning_rate)

# --- Trainer Class Definition ---
# You will be implementing methods for this class.

class Trainer:
    def __init__(self, model: nn.Module, train_dataset: torch.Tensor, config: TrainerConfig):
        self.model = model
        self.train_dataset = train_dataset
        self.config = config
        self.optimizer = model.configure_optimizers(config)
        self.iter_num = 0

    def get_batch(self) -> tuple[torch.Tensor, torch.Tensor]:
        """
        Pulls a random batch of data from the training set.

        This method should sample `batch_size` random starting points from the dataset,
        and create input sequences (`x`) of length `block_size` and corresponding
        target sequences (`y`). Remember, for language modeling, the target for a
        given token is the very next token in the sequence.

        Returns:
            A tuple of (x, y) tensors.
            - x: A (batch_size, block_size) tensor of input token indices.
            - y: A (batch_size, block_size) tensor of target token indices.
        """
        # This is Exercise 1
        raise NotImplementedError

    def run_step(self, x: torch.Tensor, y: torch.Tensor) -> float:
        """
        Performs a single training step.

        This involves:
        1. Zeroing out gradients from the previous step.
        2. Performing a forward pass to get logits and loss.
        3. Performing a backward pass to compute gradients.
        4. Updating the model's parameters using the optimizer.

        Args:
            x: Input tensor of shape (batch_size, block_size).
            y: Target tensor of shape (batch_size, block_size).

        Returns:
            The loss value for this step as a float.
        """
        # This is Exercise 2
        raise NotImplementedError

    def train(self) -> None:
        """
        Runs the full training loop for `max_iters`.

        This method should loop `max_iters` times. In each iteration, it should:
        1. Get a batch of data using `get_batch`.
        2. Run a training step with that batch using `run_step`.
        3. Periodically print the loss to monitor training progress.
        """
        # This is Exercise 3
        raise NotImplementedError

### Exercise 1: Get a Batch of Data

Your first task is to implement the `get_batch` method. This function is responsible for creating mini-batches of data that will be fed to the model. For a dataset that is just one long sequence of tokens, you need to randomly select starting points and slice out chunks of data.

**Instructions:**
1.  Generate `batch_size` random integers. These will be the starting indices for your sequences in the `train_dataset`. Make sure the indices are chosen such that you can still slice a full `block_size + 1` chunk from the dataset.
2.  Use these indices to create the input tensor `x` by stacking `block_size` length sequences.
3.  Create the target tensor `y` by stacking the sequences that are shifted by one position to the right of `x`.

**Hint:** `torch.randint` is a great way to get random starting indices. You can then use list comprehensions and `torch.stack` to build your batches.

In [ ]:
def get_batch_impl(self) -> tuple[torch.Tensor, torch.Tensor]:
    """
    Pulls a random batch of data from the training set.

    This method should sample `batch_size` random starting points from the dataset,
    and create input sequences (`x`) of length `block_size` and corresponding
    target sequences (`y`). Remember, for language modeling, the target for a
    given token is the very next token in the sequence.

    Returns:
        A tuple of (x, y) tensors.
        - x: A (batch_size, block_size) tensor of input token indices.
        - y: A (batch_size, block_size) tensor of target token indices.
    """
    ### START CODE HERE ###
    # your code here
    pass
    ### END CODE HERE ###
    return x, y

# We attach the implementation to the class now. In a real scenario, you'd edit the class directly.
Trainer.get_batch = get_batch_impl

In [ ]:
# --- Test Cell for Exercise 1 ---
print("Testing Exercise 1: get_batch...")

# Setup
torch.manual_seed(1337)
test_config = TrainerConfig(batch_size=2, block_size=5)
# A simple dataset of sequential numbers
test_dataset = torch.arange(20, dtype=torch.long)
# We don't need a real model for this test
test_trainer = Trainer(model=None, train_dataset=test_dataset, config=test_config)

# Run the function
x, y = test_trainer.get_batch()

# 1. Test shapes
assert x.shape == (2, 5), f"Shape of x is incorrect. Got {x.shape}, expected (2, 5)"
assert y.shape == (2, 5), f"Shape of y is incorrect. Got {y.shape}, expected (2, 5)"

# 2. Test content relationship (y should be x shifted by 1)
expected_y = torch.stack([test_dataset[i+1:i+6] for i in [1, 11]]) # Based on seed 1337
assert torch.equal(x[:, 1:], y[:, :-1]), "Internal elements of y are not shifted version of x"
assert torch.equal(y, expected_y), f"The content of y is not what was expected. Got {y}, expected {expected_y}"

print("✅ All tests passed for get_batch!")

### Exercise 2: A Single Training Step

Great job! Now that you can get data, let's implement `run_step`. This is the heart of the training process where the model actually learns. You'll perform one full cycle of forward pass, loss calculation, backward pass, and parameter update.

**Instructions:**
1.  Set the gradients of all model parameters to zero. This is crucial to prevent gradients from accumulating across iterations.
2.  Perform the forward pass by calling the model with the inputs `x` and targets `y`. Our `DummyGPT` model is conveniently set up to return both the `logits` and the `loss`.
3.  Call `.backward()` on the loss tensor to compute the gradients for all model parameters.
4.  Tell the optimizer to update the parameters based on the computed gradients.
5.  Return the loss value. Use `.item()` to get the scalar Python number from the loss tensor.

In [ ]:
def run_step_impl(self, x: torch.Tensor, y: torch.Tensor) -> float:
    """
    Performs a single training step.

    This involves:
    1. Zeroing out gradients from the previous step.
    2. Performing a forward pass to get logits and loss.
    3. Performing a backward pass to compute gradients.
    4. Updating the model's parameters using the optimizer.

    Args:
        x: Input tensor of shape (batch_size, block_size).
        y: Target tensor of shape (batch_size, block_size).

    Returns:
        The loss value for this step as a float.
    """
    ### START CODE HERE ###
    # your code here
    pass
    ### END CODE HERE ###

    return loss.item()

Trainer.run_step = run_step_impl

In [ ]:
# --- Test Cell for Exercise 2 ---
print("Testing Exercise 2: run_step...")

# Setup
torch.manual_seed(42)
vocab_size = 10
n_embd = 4
block_size = 5
batch_size = 2

test_config = TrainerConfig(batch_size=batch_size, block_size=block_size)
test_model = DummyGPT(vocab_size=vocab_size, n_embd=n_embd, block_size=block_size)
test_dataset = torch.randint(0, vocab_size, (20,))
test_trainer = Trainer(model=test_model, train_dataset=test_dataset, config=test_config)

# Create a sample batch
x_sample = torch.randint(0, vocab_size, (batch_size, block_size))
y_sample = torch.randint(0, vocab_size, (batch_size, block_size))

# Store initial parameter values
initial_params = [p.clone() for p in test_model.parameters()]

# Run the function
loss = test_trainer.run_step(x_sample, y_sample)

# 1. Test loss type
assert isinstance(loss, float), f"Loss should be a float, but got {type(loss)}"

# 2. Test that parameters have been updated
params_updated = False
for p_init, p_final in zip(initial_params, test_model.parameters()):
    if not torch.equal(p_init, p_final):
        params_updated = True
        break
assert params_updated, "Model parameters were not updated after the training step."

# 3. Check if gradients are zeroed out for the next step
# We run another step. If gradients were not zeroed, they would accumulate and lead to an error or different results.
# This is a bit implicit, but a direct test is complex. The success of the next test relies on it.
try:
    _ = test_trainer.run_step(x_sample, y_sample)
    print("A second step ran successfully, suggesting gradients were correctly handled.")
except Exception as e:
    assert False, f"A second training step failed. Did you remember to call optimizer.zero_grad()? Error: {e}"


print("✅ All tests passed for run_step!")

### Exercise 3: The Full Training Loop

You are doing fantastic! With `get_batch` and `run_step` ready, the final piece of the puzzle is the `train` method itself. This method will orchestrate the entire process, calling your previously implemented methods in a loop.

**Instructions:**
1.  Loop from `0` up to `self.config.max_iters`.
2.  Inside the loop, get a new batch of data by calling `self.get_batch()`.
3.  Execute a training step by passing this batch to `self.run_step()`.
4.  You can also add a print statement to log the loss periodically (e.g., every 10 iterations) to see your model learn in real-time!

In [ ]:
def train_impl(self) -> None:
    """
    Runs the full training loop for `max_iters`.

    This method should loop `max_iters` times. In each iteration, it should:
    1. Get a batch of data using `get_batch`.
    2. Run a training step with that batch using `run_step`.
    3. Periodically print the loss to monitor training progress.
    """
    self.model.train() # Set model to training mode
    for i in range(self.config.max_iters):
        ### START CODE HERE ###
    # your code here
    pass
        ### END CODE HERE ###

        # 3. Log progress
        if i % 10 == 0 or i == self.config.max_iters - 1:
            print(f"Iteration {i}: loss {loss:.4f}")

        self.iter_num += 1

Trainer.train = train_impl

In [ ]:
# --- Integration Test: Wiring it all together ---
print("Running integration test...")

# Setup
torch.manual_seed(42)
vocab_size = 20
n_embd = 8
train_dataset = torch.randint(0, vocab_size, (500,))

# We expect the loss to decrease, so we'll use more iterations
config = TrainerConfig(
    max_iters=50,
    batch_size=8,
    block_size=16,
    learning_rate=3e-3
)

model = DummyGPT(vocab_size=vocab_size, n_embd=n_embd, block_size=config.block_size)
trainer = Trainer(model, train_dataset, config)

# Let's capture the first and last loss
x_first, y_first = trainer.get_batch()
torch.manual_seed(42) # Reset seed to get the same batch again
initial_loss = trainer.run_step(x_first, y_first)
print(f"Initial loss on a sample batch: {initial_loss:.4f}")

# Reset the model and trainer for a full run
torch.manual_seed(42)
model = DummyGPT(vocab_size=vocab_size, n_embd=n_embd, block_size=config.block_size)
trainer = Trainer(model, train_dataset, config)

# Run the full training loop
print("\nStarting training...")
trainer.train()
print("Training complete.")

# Check the loss on the same batch as before
final_loss_logits, final_loss_tensor = model(x_first, y_first)
final_loss = final_loss_tensor.item()
print(f"Final loss on the same sample batch: {final_loss:.4f}\n")

# Test that training has occurred and loss has decreased
assert final_loss < initial_loss, f"Loss did not decrease! Initial: {initial_loss}, Final: {final_loss}"

print("✅ Congratulations! You have successfully implemented the training loop.")
print("The loss has decreased, which means your model is learning!")

### 🥳 Challenge (Optional)

Congratulations on implementing the `Trainer`! As an optional challenge, try extending it with an evaluation mechanism.

**Goal:** Modify the `train` loop to periodically evaluate the model on a validation dataset.

**Steps:**
1.  Add a `val_dataset` argument to the `Trainer`'s `__init__` method.
2.  Create a new method, `evaluate()`, which calculates the average loss over the entire validation set.
    *   Remember to use `torch.no_grad()` to prevent gradient calculations.
    *   Set the model to evaluation mode with `model.eval()` before evaluating and back to training mode with `model.train()` afterwards.
3.  Modify your `train` loop to call `evaluate()` every, for example, 25 iterations and print both the training loss and the validation loss. This is key to monitoring for overfitting